# P-2026-11-27-1: Does a de-duplicated RCA denominator improve the lab's grouped ECI?

**Question** (from `proposals/2026-11-25.md` #2, in the open 11-25 PR): the lab's `compute_mcp` (`code/current/Spring2026.ipynb` #11) computes each country's total and the world total over **all 18 columns**. But *High-complexity Products* is an aggregate of chapters that already sit in other columns (28–30, 38, 83, 84 partly, 87, 92, 99), so those dollars are counted twice in every denominator. What if we keep the aggregate column but compute the totals over the **17 non-aggregate columns only** (DD)? How many M cells flip, and does grouped ECI get better under the pre-stated rule A?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`. Open nightly PRs for 11-20 to 11-26 each took one idea from the previous night's proposals. 11-25 #1 was taken by 11-26. **11-25 #2 is untaken**: no run folder on `main` or on any `claude/nightly-*` branch names it. It is the cheapest change to the lab's own code on the list (one line in `compute_mcp`), and it isolates the denominator effect that 11-25 (C3) and 11-26 (E) both found to drive most of their rank changes.

- **Date:** 2026-11-27 (America/New_York), nightly run. ID `P-2026-11-27-1`.
- **Data:** `data/cleaned/2019–2023clean.csv` (the cleaned grouped base, via `lib/research.lab_groups_iso3`, EU-28 dropped). `data/raw/2019–2023_mega.csv` (the lab mapping rebuilt with `research.groups_from_raw` as a robustness base, plus the HS4 matrix that defines the sample). `external-data/raw/atlas_hs92_country_year.csv` (ATLAS ECI, a benchmark only). `external-data/raw/wdi_panel.csv` (GDPpc PPP). No downloads.
- **Years:** 2019 primary. 2020–2022 are checks. 2023 is shown but suspect (D1).
- **Library:** `lib/eci.py` and `lib/research.py`, imported unchanged. `o4`, the lab filter and the joint common sample are copied from 11-24/11-25/11-26, so the numbers are directly comparable.
- **Run time:** under 20 s (printed in the last cell).

## TL;DR

- **The double count is big.** In the 2019 cleaned data, the aggregate makes the world total over 18 columns **$21,377B**, against **$15,575B** over the 17 real columns, a **37.2%** inflation (33.6–37.2% across 2019–2023). Algebraically, DD multiplies *every* RCA of country *c* by one factor, (1 + a_c)/(1 + a_w), where a is the aggregate's share. This was verified to machine precision. The 2019 factor ranges from **0.732** (Angola) to **1.363** (Saudi Arabia), with a median of **0.832**. So the lab's RCA > 1 threshold is effectively looser for the typical country and stricter for aggregate-heavy exporters (Germany 1.156, Japan 1.212).
- **DD passes rule A on the pre-stated primary base (cleaned CSV) but not on the rebuilt base, so it is a weak pass.** Cleaned: ΔM1 **+0.051** [−0.010, 0.115], ΔM2 **+0.023** [−0.068, 0.113] in 2019, and both are up in **2 of 3** check years (2020, 2021; 2022 has ΔM1 −0.003). Rebuilt from raw: 2019 ΔM2 is **−0.006**, so it fails, though both metrics are still up in 2020 and 2021. All 2019 bootstrap intervals include 0. Across both bases and 2019–2022, **14 of 16** metric differences are positive.
- **DD is exactly "A17 cells + the lab's aggregate column", and that beats dropping the aggregate.** DD's 17 non-aggregate RCAs are identical to A17's (max difference 0.0), and 0 aggregate cells flip against D0. Keeping the aggregate column, with honest denominators, beats dropping it (A17) on M2 in **every** year on both bases (+0.027 to +0.105). A17 itself fails rule A (1 of 3 check years).
- **Many cells move, mostly downward.** 2019: **80** M cells flip (65 1→0, 15 0→1) across **57** of 120 countries. Every country that gains a cell has factor > 1, and every one that loses one has factor < 1. Group PCI order barely changes (Spearman 0.988–0.996). The biggest 2019 moves: Kuwait and Brunei 59 → 103 (each loses its *Chemicals* cell), Russia 77 → 46, UK 37 → 21, Vietnam 32 → 54, Moldova 25 → 50.
- **For the reports:** 6 of 28 report-country peer positions change in 2019–2022 (UAE 3 → 2 in 2019/2020, UK 4 → 3 in 2019 and 3 → 2 in 2021, Vietnam 2 → 3 in 2020 and 3 → 2 in 2022). Adopting DD is a real change to printed numbers, not a cosmetic one. Will's call, and the evidence is base-dependent.

## Why this matters for the lab

Every country report prints a grouped-ECI rank and compares it with three peers. Every one of them goes through `compute_mcp`. The aggregate column is intentional (KNOWN-ISSUES D1 and methodology Table 1 say it overlaps other groups *by design*). But nobody has checked what the overlap does to the **denominators** of the RCA, which every other column depends on. 11-25 found that removing ch. 99 from the aggregate flipped 18 cells *outside* the aggregate column through the world total, and 11-26 found the same thing with Saudi Arabia's reassignment. Tonight tests the general fix: stop the aggregate inflating the totals, and change nothing else. It is a one-line change to the lab's own implementation (agenda section 3), it doesn't touch the cleaning mapping, and it is scored under the same rule A as every variant since 11-19 (agenda section 5).

## Method (pre-stated)

**Base rules:** the lab's grouped ECI (the drop-any-zero filter applied to the 18-column matrix, so **all variants use the same countries**; strict `RCA > 1`; sorted SVD; O4 orientation) on the **joint common sample** of 11-21 to 11-26 (cleaned grouped base ∩ HS4 base ∩ ATLAS ∩ WDI ∩ resource share: 105 countries in 2019).

**Two bases.** The primary base is **C = the cleaned CSV**, which is the file `code/current/Spring2026.ipynb` actually reads, and a `compute_mcp` change applies to it directly. The robustness base is **R = the lab mapping rebuilt from `data/raw`**, the base 11-24 to 11-26 used for mapping variants. The two differ in 2019–2021 by the known cleaned-vs-raw gaps (Q05).

**Variants** (only the RCA formula changes):
- **D0** = lab `compute_mcp`: totals over all 18 columns (the baseline).
- **DD** = de-duplicated denominator: the same 18 columns, but country totals and the world total are summed over the 17 non-aggregate columns. The aggregate's own RCA uses the same honest denominators. **Primary variant.**
- **A17** = aggregate column dropped (Q02 / Julia's 17-group choice). Reference only.

**Metrics and rule** (unchanged since 11-19): **M1** = Spearman with ATLAS ECI. **M2** = Spearman with log GDPpc (PPP). **Rule A:** a variant is *better* only if it beats D0 on both M1 and M2 in 2019 **and** in ≥ 2 of 2020–2022. The 2019 paired country bootstrap uses 2,000 draws. Report only: M3 = Spearman with our HS4 ECI, and the year-to-year rank stability on a fixed panel.

**Pre-stated diagnostics:** (A) the size of the double count and an exact check of the analytic factor; (B) M flips by column and direction; (C3) DD vs A17 head to head; (C4) rule A with the countries whose M row changes left out of the Spearman; (D) movers and group PCI; (E) Spring 2026 report countries and their position among their 3 peers (the same peer dict as Q02 / 11-14 / 11-17 / 11-25 / 11-26).

**Transparency note:** before writing this notebook, a scratch prototype computed DD's M1/M2 on the cleaned base only. The cleaned base was named primary at that point, because it is the notebook's real input. The rebuilt base was then added as the pre-stated robustness check, and its result (a fail) is reported with equal weight below.

## Setup

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2019, 2020, 2021, 2022]
AGG = 'High-complexity Products'
PEERS = {'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
         'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'], 'VNM': ['SGP', 'MYS', 'PHL']}

def o4(M):
    """Sorted SVD of binary M, O4 orientation (PCI negative with ubiquity). Copied unchanged from 11-24/11-25/11-26."""
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)
    s4 = -np.sign(np.corrcoef(q, kp)[0, 1]); k, q = k * s4, q * s4
    e = pd.Series(k, index=M.index); z = (e - e.mean()) / e.std()
    return z, e, pd.Series(q, index=M.columns), S[1]

# ---- the three RCA definitions (the only thing that differs between variants) ----
def rca_lab(X):
    """D0: lab compute_mcp denominators -- country and world totals over all 18 columns (aggregate double-counted)."""
    return eci.rca(X)

def rca_dedup(X):
    """DD: same 18 columns, but country and world totals summed over the 17 NON-aggregate columns only."""
    X = X.loc[X.sum(1) > 0, X.sum(0) > 0]; base = X.drop(columns=[AGG])
    return X.div(base.sum(1), axis=0) / (X.sum(0) / base.to_numpy().sum())

def rca_drop(X):
    """A17: aggregate column dropped entirely (Q02 / Julia's 17-group choice). Reference variant."""
    return eci.rca(X.drop(columns=[AGG]))

VARS = {'D0 lab denominator (18 cols)': rca_lab, 'DD de-duplicated denominator (primary)': rca_dedup,
        'A17 aggregate dropped (ref.)': rca_drop}
VN = list(VARS); V0 = VN[0]; VD = VN[1]; VA = VN[2]

def lab_M(X, f):
    """Lab rules: drop-any-zero filter on the 18-column matrix (same sample for every variant), strict RCA > 1."""
    X = eci.notebook_filter(X); Mm = (f(X) > 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def hs_M(X):
    Mm = (eci.rca(X) >= 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def rank(e, idx):
    return e[idx].rank(ascending=False)

# Bases: C = cleaned CSV (what code/current/Spring2026.ipynb reads; PRIMARY). R = lab mapping rebuilt from data/raw (as 11-24..11-26).
BASE = {'C': {y: R.lab_groups_iso3(y, drop_eu=True) for y in YEARS},
        'R': {y: R.groups_from_raw(y, R.LAB_GROUPS) for y in YEARS}}
XH = {y: R.repo_hs(y, 4) for y in YEARS}
RSH = {y: XH[y][[c for c in XH[y].columns if c[:2] in ['26', '27', '71']]].sum(1) / XH[y].sum(1) for y in YEARS}
AT = R.atlas_cy(); ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
GB = {y: o4(lab_M(BASE['C'][y], rca_lab)) for y in YEARS}; HB = {y: o4(hs_M(XH[y])) for y in YEARS}
COMMON = {y: sorted(set(GB[y][0].index) & set(HB[y][0].index) & set(ATL[y].dropna().index) & set(LNY[y].index) & set(RSH[y].index)) for y in YEARS}
print('joint common sample n (same definition as 11-21/11-24/11-25/11-26):', {y: len(COMMON[y]) for y in YEARS})

MM, VAR, PCI = {}, {}, {}
for b in BASE:
    for y in YEARS:
        for v, f in VARS.items():
            MM[b, y, v] = lab_M(BASE[b][y], f); z, k, q, s_ = o4(MM[b, y, v]); VAR[b, y, v] = z; PCI[b, y, v] = q
CS = {(b, y, v): sorted(set(COMMON[y]) & set(VAR[b, y, V0].index) & set(VAR[b, y, v].index)) for b in BASE for y in YEARS for v in VN}
print('pairwise sample n, cleaned base:', {y: [len(CS['C', y, v]) for v in VN] for y in YEARS})
print('pairwise sample n, rebuilt base:', {y: [len(CS['R', y, v]) for v in VN] for y in YEARS})


joint common sample n (same definition as 11-21/11-24/11-25/11-26): {2019: 105, 2020: 105, 2021: 94, 2022: 105, 2023: 109}
pairwise sample n, cleaned base: {2019: [105, 105, 105], 2020: [105, 105, 105], 2021: [94, 94, 94], 2022: [105, 105, 105], 2023: [109, 109, 109]}
pairwise sample n, rebuilt base: {2019: [105, 105, 105], 2020: [105, 105, 105], 2021: [94, 94, 94], 2022: [105, 105, 105], 2023: [109, 109, 109]}


**Takeaway:** the sample matches 11-24 to 11-26 exactly (105 / 105 / 94 / 105 / 109). Every variant uses the same countries in every year on both bases.

## Results

### A. How big is the double count, and what does de-duplication do to each RCA?

In [2]:
# A. How big is the double count, and what does de-duplication do to each RCA? (cleaned base)
rows = []
for y in YEARS:
    X = eci.notebook_filter(BASE['C'][y]); T18, T17 = X.sum(1), X.drop(columns=[AGG]).sum(1)
    a_c = X[AGG] / T17; a_w = X[AGG].sum() / T17.sum()
    fac = (1 + a_c) / (1 + a_w)                               # analytic: RCA_DD / RCA_D0 for EVERY column of country c
    ratio = rca_dedup(X) / rca_lab(X)
    same17 = (rca_dedup(X).drop(columns=[AGG]) - rca_drop(X)).abs().to_numpy().max()
    rows.append({'year': y, 'n (after filter)': len(X), 'world 18-col $B': X.to_numpy().sum() / 1e9, 'world 17-col $B': T17.sum() / 1e9,
                 'inflation of world total': X.to_numpy().sum() / T17.sum() - 1, 'world agg/17-col (a_w)': a_w,
                 'max |ratio - analytic factor|': (ratio.sub(fac, axis=0)).abs().to_numpy().max(),
                 'max |DD - A17| on 17 cols': same17,
                 'factor min': fac.min(), 'factor median': fac.median(), 'factor max': fac.max()})
A = pd.DataFrame(rows); print(A.round(4).to_string(index=False))
y = 2019; X = eci.notebook_filter(BASE['C'][y]); T17 = X.drop(columns=[AGG]).sum(1)
a_c = X[AGG] / T17; a_w = X[AGG].sum() / T17.sum(); fac = ((1 + a_c) / (1 + a_w)).sort_values()
print('\n2019 factor RCA_DD/RCA_D0, lowest 5 :', ', '.join(f'{i} {v:.3f}' for i, v in fac.head(5).items()))
print('2019 factor RCA_DD/RCA_D0, highest 5:', ', '.join(f'{i} {v:.3f}' for i, v in fac.tail(5).items()))
print('2019 report countries:', ', '.join(f'{i} {fac[i]:.3f}' for i in R.REPORT_COUNTRIES if i in fac.index))


 year  n (after filter)  world 18-col $B  world 17-col $B  inflation of world total  world agg/17-col (a_w)  max |ratio - analytic factor|  max |DD - A17| on 17 cols  factor min  factor median  factor max
 2019               120        21376.526        15574.639                     0.372                   0.372                            0.0                        0.0       0.732          0.832       1.363
 2020               120        19907.354        14557.782                     0.367                   0.367                            0.0                        0.0       0.733          0.856       1.320
 2021               105        24715.794        18392.893                     0.344                   0.344                            0.0                        0.0       0.746          0.871       1.339
 2022               120        26996.662        20212.640                     0.336                   0.336                            0.0                        0.0       0.750   

**Takeaway (A):** the aggregate inflates the world total by **37.2%** in 2019 ($21,377B vs $15,575B over the 17 real columns; 33.6–37.2% in other years). The ratio RCA_DD / RCA_D0 equals the analytic per-country factor (1 + a_c)/(1 + a_w) exactly (max error 0.0), and DD's 17 non-aggregate RCAs are identical to A17's. The 2019 factor runs from 0.732 (Angola) to 1.363 (Saudi Arabia, whose 999999 oil sits in the aggregate; see 11-25), with a median of 0.832. Report countries: Germany 1.156 and UK 1.091 (their RCAs rise under DD), Vietnam 0.798 and UAE 0.797 (theirs fall). In other words, the lab's D0 threshold "RCA > 1" means "DD-RCA > 0.832" for the median country but "DD-RCA > 1.156" for Germany.

### B. Which M cells flip?

In [3]:
# B. Which M cells flip, D0 -> DD and D0 -> A17 (cleaned base)
rows = []
for y in YEARS:
    for v in [VD, VA]:
        A0, A1 = MM['C', y, V0], MM['C', y, v]; idx = sorted(set(A0.index) & set(A1.index)); cols = sorted(set(A0.columns) & set(A1.columns))
        D = (A0.loc[idx, cols] != A1.loc[idx, cols])
        up = ((A1.loc[idx, cols] == 1) & (A0.loc[idx, cols] == 0)).to_numpy().sum()
        rows.append({'year': y, 'variant': v.split()[0], 'cells': D.size, 'flips': int(D.to_numpy().sum()), '0->1': int(up), '1->0': int(D.to_numpy().sum() - up),
                     'flips in agg col': int(D[AGG].sum()) if AGG in cols else '-', 'countries with a flip': int(D.any(axis=1).sum()),
                     'D0 diversity mean': A0.loc[idx].sum(1).mean(), 'variant diversity mean': A1.loc[idx].sum(1).mean()})
FLP = pd.DataFrame(rows); print(FLP.round(2).to_string(index=False))
y = 2019; A0, A1 = MM['C', y, V0], MM['C', y, VD]; idx = sorted(set(A0.index) & set(A1.index))
D = (A0.loc[idx] != A1.loc[idx]); byc = D.sum().sort_values(ascending=False)
print('\n2019 DD flips by column:', ', '.join(f'{c} {int(n)}' for c, n in byc[byc > 0].items()))
# Direction: are 0->1 flips concentrated in countries with factor > 1 (above-average aggregate share)?
X = eci.notebook_filter(BASE['C'][y]); T17 = X.drop(columns=[AGG]).sum(1); fac = (1 + X[AGG] / T17) / (1 + X[AGG].sum() / T17.sum())
up_c = [i for i in idx if ((A1.loc[i] == 1) & (A0.loc[i] == 0)).any()]; dn_c = [i for i in idx if ((A1.loc[i] == 0) & (A0.loc[i] == 1)).any()]
print(f'2019: countries gaining a cell: {len(up_c)}, all with factor > 1: {all(fac[up_c] > 1)} | losing a cell: {len(dn_c)}, all with factor < 1: {all(fac[dn_c] < 1)}')


 year variant  cells  flips  0->1  1->0 flips in agg col  countries with a flip  D0 diversity mean  variant diversity mean
 2019      DD   2160     80    15    65                0                     57               5.93                    5.52
 2019     A17   2040     80    15    65                -                     57               5.93                    5.32
 2020      DD   2160     70    12    58                0                     54               5.84                    5.46
 2020     A17   2040     70    12    58                -                     54               5.84                    5.25
 2021      DD   1890     69    18    51                0                     50               5.91                    5.60
 2021     A17   1785     69    18    51                -                     50               5.91                    5.36
 2022      DD   2160     63    14    49                0                     48               5.76                    5.47
 2022     A17   

**Takeaway (B):** in 2019, **80** of 2,160 cells flip (65 go 1→0, 15 go 0→1) across **57** countries. That is 70 / 69 / 63 / 74 in 2020–2023. **0** flips are in the aggregate column, and DD and A17 flip exactly the same cells. Flips spread over 17 columns, led by *Raw Manufacturing* (12) and *Daily Instruments* (11). The direction is fully determined by the factor: all 11 countries that gain a cell have factor > 1, and all 46 that lose one have factor < 1. Mean diversity falls from 5.93 to 5.52 groups.

### C. Rule A on the joint common sample

In [4]:
# C. Rule A on the joint common sample, both bases (pre-stated, unchanged since 11-19)
def metrics(b):
    rows = []
    for y in YEARS:
        for v in VN:
            c = CS[b, y, v]; e, e0 = VAR[b, y, v][c], VAR[b, y, V0][c]
            m1, m2 = stats.spearmanr(e, ATL[y][c])[0], stats.spearmanr(e, LNY[y][c])[0]
            rows.append({'base': b, 'year': y, 'variant': v, 'n': len(c), 'M1 rho ATLAS': m1, 'M2 rho lnGDPpc': m2,
                         'M1 diff': m1 - stats.spearmanr(e0, ATL[y][c])[0], 'M2 diff': m2 - stats.spearmanr(e0, LNY[y][c])[0],
                         'rho w/ D0': stats.spearmanr(e, e0)[0], 'M3 rho HS4 (report only)': stats.spearmanr(e, HB[y][0][c])[0]})
    return pd.DataFrame(rows)
P = pd.concat([metrics('C'), metrics('R')], ignore_index=True)
print(P.round(3).to_string(index=False))
P.round(4).to_csv(FIG / 'primary_metrics.csv', index=False)


base  year                                variant   n  M1 rho ATLAS  M2 rho lnGDPpc  M1 diff  M2 diff  rho w/ D0  M3 rho HS4 (report only)
   C  2019           D0 lab denominator (18 cols) 105         0.663           0.425    0.000    0.000      1.000                     0.729
   C  2019 DD de-duplicated denominator (primary) 105         0.714           0.448    0.051    0.023      0.933                     0.736
   C  2019           A17 aggregate dropped (ref.) 105         0.708           0.421    0.045   -0.004      0.922                     0.729
   C  2020           D0 lab denominator (18 cols) 105         0.762           0.490    0.000    0.000      1.000                     0.776
   C  2020 DD de-duplicated denominator (primary) 105         0.769           0.518    0.007    0.028      0.954                     0.778
   C  2020           A17 aggregate dropped (ref.) 105         0.764           0.484    0.002   -0.007      0.947                     0.767
   C  2021           D0 lab

**Takeaway (C):** on the cleaned base, DD raises M1 0.663 → 0.714 and M2 0.425 → 0.448 in 2019. It is up on both metrics in 2020 (+0.007 / +0.028) and 2021 (+0.031 / +0.059), and in 2022 M1 slips by 0.003 while M2 rises by 0.011. On the rebuilt base, the 2019 M1 rises (+0.027) but M2 slips (**−0.006**). 2020 and 2021 are again up on both. 2022–2023 are identical on the two bases (the cleaned and raw data agree there). M3 (HS4 agreement) rises with DD in 2019–2021 on the cleaned base (0.729 → 0.736, 0.776 → 0.778, 0.810 → 0.844). A17 loses income fit in 2019 on both bases and badly in 2022 (−0.094).

In [5]:
# C2. Rule A verdicts + 2019 paired country bootstrap (2,000 draws), both bases
rng = np.random.default_rng(20261127); y = 2019; B = 2000
dec = []
for b in BASE:
    for v in VN[1:]:
        c = np.array(CS[b, y, v]); at, ly = ATL[y][c].to_numpy(), LNY[y][c].to_numpy()
        e0, e1 = VAR[b, y, V0][c].to_numpy(), VAR[b, y, v][c].to_numpy()
        bb = np.array([(stats.spearmanr(e1[i], at[i])[0] - stats.spearmanr(e0[i], at[i])[0],
                        stats.spearmanr(e1[i], ly[i])[0] - stats.spearmanr(e0[i], ly[i])[0]) for i in rng.integers(0, len(c), size=(B, len(c)))])
        s = P[(P.base == b) & (P.variant == v)].set_index('year'); q1, q2 = np.percentile(bb[:, 0], [2.5, 97.5]), np.percentile(bb[:, 1], [2.5, 97.5])
        both = (s['M1 diff'] > 0) & (s['M2 diff'] > 0)
        dec.append({'base': b, 'variant': v, 'M1 diff 2019': round(s.loc[2019, 'M1 diff'], 3), 'M1 95% CI': f'[{q1[0]:.3f}, {q1[1]:.3f}]',
                    'M2 diff 2019': round(s.loc[2019, 'M2 diff'], 3), 'M2 95% CI': f'[{q2[0]:.3f}, {q2[1]:.3f}]',
                    'both 2019': bool(both[2019]), 'both, of 2020-22': int(both.loc[[2020, 2021, 2022]].sum()),
                    'which check yrs': ','.join(str(t) for t in [2020, 2021, 2022] if both[t]),
                    'BETTER (rule A)': bool(both[2019] and both.loc[[2020, 2021, 2022]].sum() >= 2)})
DEC = pd.DataFrame(dec); print(DEC.to_string(index=False))


base                                variant  M1 diff 2019       M1 95% CI  M2 diff 2019       M2 95% CI  both 2019  both, of 2020-22 which check yrs  BETTER (rule A)
   C DD de-duplicated denominator (primary)         0.051 [-0.010, 0.115]         0.023 [-0.068, 0.113]       True                 2       2020,2021             True
   C           A17 aggregate dropped (ref.)         0.045 [-0.018, 0.112]        -0.004 [-0.098, 0.086]      False                 1            2021            False
   R DD de-duplicated denominator (primary)         0.027 [-0.038, 0.097]        -0.006 [-0.097, 0.092]      False                 2       2020,2021            False
   R           A17 aggregate dropped (ref.)         0.019 [-0.051, 0.091]        -0.038 [-0.138, 0.054]      False                 1            2021            False


**Takeaway (C2): DD passes rule A on the cleaned base (2019 plus 2020 and 2021) and fails on the rebuilt base (2019 ΔM2 −0.006).** No 2019 bootstrap interval excludes 0, for any variant on either base. A17 fails on both bases. So the honest verdict is a **weak, base-dependent pass**. The direction is consistent (14 of 16 base × year × metric differences in 2019–2022 are positive), but the margin is inside the noise.

In [6]:
# C3. Pre-stated robustness: DD vs A17 directly (does keeping the aggregate column, with honest denominators, add anything?)
rows = []
for b in BASE:
    for y in YEARS:
        c = sorted(set(CS[b, y, VD]) & set(CS[b, y, VA])); e, e0 = VAR[b, y, VD][c], VAR[b, y, VA][c]
        rows.append({'base': b, 'year': y, 'n': len(c), 'M1 diff DD-A17': stats.spearmanr(e, ATL[y][c])[0] - stats.spearmanr(e0, ATL[y][c])[0],
                     'M2 diff DD-A17': stats.spearmanr(e, LNY[y][c])[0] - stats.spearmanr(e0, LNY[y][c])[0], 'rho DD vs A17': stats.spearmanr(e, e0)[0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
# C4. Pre-stated robustness: leave out the countries whose M row changes under DD (their ranks still shape everyone's ECI)
rows = []
for y in CHK:
    A0, A1 = MM['C', y, V0], MM['C', y, VD]; idx = sorted(set(A0.index) & set(A1.index))
    ch = [i for i in idx if (A0.loc[i] != A1.loc[i]).any()]
    c = [i for i in CS['C', y, VD] if i not in ch]; e, e0 = VAR['C', y, VD][c], VAR['C', y, V0][c]
    rows.append({'year': y, 'excluded (row changes)': len(ch), 'n': len(c),
                 'M1 diff': stats.spearmanr(e, ATL[y][c])[0] - stats.spearmanr(e0, ATL[y][c])[0],
                 'M2 diff': stats.spearmanr(e, LNY[y][c])[0] - stats.spearmanr(e0, LNY[y][c])[0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))


base  year   n  M1 diff DD-A17  M2 diff DD-A17  rho DD vs A17
   C  2019 105           0.006           0.027          0.984
   C  2020 105           0.005           0.034          0.980
   C  2021  94          -0.007           0.036          0.981
   C  2022 105           0.015           0.105          0.950
   C  2023 109           0.013           0.056          0.982
   R  2019 105           0.008           0.032          0.985
   R  2020 105           0.006           0.035          0.981
   R  2021  94           0.007           0.052          0.975
   R  2022 105           0.015           0.105          0.950
   R  2023 109           0.013           0.056          0.982
 year  excluded (row changes)  n  M1 diff  M2 diff
 2019                      57 55    0.016    0.014
 2020                      54 59   -0.012    0.003
 2021                      50 49    0.017    0.041
 2022                      48 62    0.010    0.015


**Takeaway (C3/C4):** DD beats A17 on M2 in every year on both bases (+0.027 to +0.105), and on M1 in 9 of 10 base-years. That is the cleanest result tonight: *if the lab keeps the aggregate column, it should stop letting it inflate the denominators*, and keeping it is better than dropping it. Leaving out the 57 countries whose M row changes (C4), DD's 2019 effect on the other 55 countries' ordering is still positive on both metrics (+0.016 / +0.014). 2021 and 2022 are positive too, and 2020 has M1 −0.012. So the gain isn't only the flipped countries correcting themselves.

### D. Movers and group PCI

In [7]:
# D. Movers and group PCI (cleaned base)
for y in CHK:
    c = CS['C', y, VD]; d = (rank(VAR['C', y, VD], c) - rank(VAR['C', y, V0], c)).sort_values()
    big = d[d.abs() >= 10]
    print(y, f'| n moving >= 5 ranks: {int((d.abs() >= 5).sum())} of {len(c)} | median |move| {d.abs().median():.1f} | >= 10:',
          ', '.join(f'{i} {int(rank(VAR["C", y, V0], c)[i])}->{int(rank(VAR["C", y, VD], c)[i])}' for i in big.index) or '-')
y = 2019
pr = pd.DataFrame({v.split()[0]: PCI['C', y, v].rank(ascending=False) for v in VN}).astype('Int64').sort_values('D0')
print('\n2019 group PCI rank (1 = most complex):'); print(pr.to_string())
print('Spearman of group PCI, D0 vs DD (18 groups):', {yy: round(stats.spearmanr(PCI['C', yy, V0], PCI['C', yy, VD][PCI['C', yy, V0].index])[0], 3) for yy in YEARS})


2019 | n moving >= 5 ranks: 59 of 105 | median |move| 5.0 | >= 10: RUS 77->46, GBR 37->21, SWE 31->15, HRV 48->33, MOZ 88->73, CYP 54->40, CAN 95->83, LUX 58->47, UZB 82->72, FIN 24->14, ZMB 38->48, MLT 40->51, PAN 17->29, HKG 12->26, NGA 20->36, KEN 49->66, PAK 61->79, GUY 78->96, TUN 19->39, VNM 32->54, BHR 69->92, MDA 25->50, COG 43->74, BRN 59->103, KWT 59->103
2020 | n moving >= 5 ranks: 47 of 105 | median |move| 4.0 | >= 10: ZMB 77->53, BIH 33->15, MDG 94->78, CYP 54->39, BRN 69->57, POL 37->27, USA 42->32, KGZ 87->97, KEN 68->79, LBN 59->71, PAK 57->70, HKG 19->33, COL 83->98, VNM 26->46, TUN 25->45, KWT 69->104, NGA 55->104
2021 | n moving >= 5 ranks: 40 of 94 | median |move| 4.0 | >= 10: RUS 71->56, UKR 75->61, LUX 53->40, KEN 54->64, LVA 43->54, OMN 41->52, MMR 69->84, VNM 29->49, MDA 49->77, TUN 18->47
2022 | n moving >= 5 ranks: 29 of 105 | median |move| 2.0 | >= 10: PRY 72->58, MDA 92->79, ZMB 58->47, UKR 53->64, URY 62->74, HRV 48->61, PAK 59->72, MMR 74->92, MYS 31->50



**Takeaway (D):** ranks move a lot. In 2019, 59 of 105 countries move ≥ 5 places (median 5), falling to 29 of 105 (median 2) in 2022. The ≥ 10-place moves mostly go **toward** ATLAS (Russia 77 → 46, ATLAS 40; UK 37 → 21, ATLAS 11; Congo 43 → 74, ATLAS 101; Moldova 25 → 50, ATLAS 72; Tunisia 19 → 39, ATLAS 57). Some go away from it (Kuwait and Brunei 59 → 103, ATLAS 60 and 49; Germany 1 → 8). Group PCI order is almost unchanged (Spearman 0.988–0.996 vs D0). The aggregate stays 2nd, and *Machinery* and *Daily Instruments* swap 3rd and 4th.

In [8]:
# D2. Why the biggest 2019 movers move: their flipped cells and RCAs (cleaned base)
y = 2019; X = eci.notebook_filter(BASE['C'][y]); r0, r1 = rca_lab(X), rca_dedup(X)
T17 = X.drop(columns=[AGG]).sum(1); fac = (1 + X[AGG] / T17) / (1 + X[AGG].sum() / T17.sum())
c = CS['C', y, VD]; d = (rank(VAR['C', y, VD], c) - rank(VAR['C', y, V0], c))
rows = []
for i in d.abs().sort_values(ascending=False).index[:8]:
    fl = [g for g in r0.columns if (r0.at[i, g] > 1) != (r1.at[i, g] > 1)]
    rows.append({'iso3': i, 'rank D0 -> DD': f'{int(rank(VAR["C", y, V0], c)[i])} -> {int(rank(VAR["C", y, VD], c)[i])}', 'ATLAS rank': int(rank(ATL[y], c)[i]),
                 'agg share of 17-col exports': X.at[i, AGG] / T17[i], 'factor': fac[i],
                 'flipped cells (RCA D0 -> DD)': '; '.join(f'{g} {r0.at[i, g]:.3f}->{r1.at[i, g]:.3f}' for g in fl)})
print(pd.DataFrame(rows).round(3).to_string(index=False))


iso3 rank D0 -> DD  ATLAS rank  agg share of 17-col exports  factor                                                           flipped cells (RCA D0 -> DD)
 BRN     59 -> 103          49                        0.059   0.771                                                                 Chemicals 1.258->0.970
 KWT     59 -> 103          60                        0.056   0.770                                                                 Chemicals 1.076->0.828
 COG      43 -> 74         101                        0.017   0.741                                                         Daily Instruments 1.010->0.749
 RUS      77 -> 46          40                        0.186   0.864               Vegetables, Fruits and Plants 1.045->0.903; Precious Stones 1.027->0.888
 MDA      25 -> 50          72                        0.065   0.776                                                            Transportation 1.036->0.803
 BHR      69 -> 92          46                        0.091   0.795   

**Takeaway (D2):** the largest moves are single-threshold events. Kuwait (*Chemicals* 1.076 → 0.828) and Brunei (1.258 → 0.970) each lose their only manufactured cell and drop 44 places. Congo loses *Daily Instruments* (1.010 → 0.749), and Moldova loses *Transportation* (1.036 → 0.803). Vietnam loses *Raw Manufacturing* (1.142 → 0.911) and *Daily Instruments* (1.181 → 0.942) and goes 32 → 54 (ATLAS 56). Russia is the reverse case: it loses two low-PCI cells (*Vegetables*, *Precious Stones*) and *rises* 77 → 46. This is the near-threshold fragility of 11-23, triggered here by a uniform per-country rescaling.

### E. Spring 2026 report countries

In [9]:
# E. Spring 2026 report countries: rank and position among (country + 3 peers), cleaned base, 2019-2022
rows = []
for y in CHK:
    for iso in R.REPORT_COUNTRIES:
        r = {'year': y, 'country': iso}
        for v in VN:
            cv = CS['C', y, v]; grp = [i for i in [iso] + PEERS[iso] if i in cv]
            t = v.split()[0]
            r[t + ' rank'] = int(rank(VAR['C', y, v], cv)[iso]) if iso in cv else None
            r[t + ' pos'] = int(VAR['C', y, v][grp].rank(ascending=False)[iso]) if iso in grp else None
        r['ATLAS rank'] = int(rank(ATL[y], CS['C', y, V0])[iso]) if iso in CS['C', y, V0] else None
        r['peers in sample'] = len([i for i in PEERS[iso] if i in CS['C', y, V0]])
        rows.append(r)
RP = pd.DataFrame(rows); print(RP.to_string(index=False))
for v in [VD, VA]:
    t = v.split()[0]; ok = RP.dropna(subset=['D0 pos', t + ' pos'])
    print(f'{t}: peer position changed in {int((ok["D0 pos"] != ok[t + " pos"]).sum())} of {len(ok)} country-years')


 year country  D0 rank  D0 pos  DD rank  DD pos  A17 rank  A17 pos  ATLAS rank  peers in sample
 2019     CAN       95       3       83       3        83        3          33                3
 2019     DEU        1       1        8       1         6        1           5                3
 2019     POL       34       4       27       4        16        4          24                3
 2019     SGP       26       4       23       4        22        4           4                3
 2019     ARE      103       3      102       2       104        3          42                2
 2019     GBR       37       4       21       3        32        4          11                3
 2019     VNM       32       3       54       3        47        3          56                2
 2020     CAN       74       3       66       3        66        3          34                3
 2020     DEU        4       1        7       1         4        1           5                3
 2020     POL       37       4       27 

**Takeaway (E):** under DD, **6 of 28** report-country peer positions change in 2019–2022: UAE 3 → 2 (2019, 2020), UK 4 → 3 (2019) and 3 → 2 (2021), Vietnam 2 → 3 (2020) and 3 → 2 (2022). Vietnam's rank moves closest to ATLAS (2019: 32 → 54, ATLAS 56; 2020: 26 → 46, ATLAS 52). The UK moves toward ATLAS in 2019–2021 (37 → 21, 29 → 20, 20 → 18) but away in 2022 (12 → 21, ATLAS 7). Germany falls from 1st to 5th–8th overall but stays 1st among its peers in every year. A17 changes 5 of 28.

### F. Secondary (report only): rank stability

In [10]:
# F. Secondary, report-only: year-to-year rank stability on a fixed panel (countries in every 2019-2022 common sample)
PAN = sorted(set.intersection(*[set(CS['C', y, v]) for y in CHK for v in VN]))
rows = []
for v in VN:
    r = {'variant': v.split()[0], 'panel n': len(PAN)}
    for a, b2 in zip(CHK[:-1], CHK[1:]):
        r[f'{a}->{b2}'] = stats.spearmanr(VAR['C', a, v][PAN], VAR['C', b2, v][PAN])[0]
    r['mean'] = np.mean([r[k] for k in r if '->' in k]); rows.append(r)
print(pd.DataFrame(rows).round(3).to_string(index=False))


variant  panel n  2019->2020  2020->2021  2021->2022  mean
     D0       86       0.925       0.943       0.962 0.943
     DD       86       0.919       0.957       0.967 0.948
    A17       86       0.912       0.959       0.943 0.938


**Takeaway (F, report only):** on the 86-country fixed panel, mean year-to-year Spearman is 0.943 (D0), **0.948 (DD)** and 0.938 (A17). DD is not less stable than the baseline.

### Figure

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
COL = {VD: '#2a78d6', VA: '#9a9994'}
for ax, m, t in zip(axes, ['M1 diff', 'M2 diff'], ['ΔM1 vs D0 (Spearman with ATLAS ECI)', 'ΔM2 vs D0 (Spearman with log GDPpc)']):
    for v in VN[1:]:
        for b, ls in [('C', '-'), ('R', ':')]:
            s = P[(P.base == b) & (P.variant == v)]
            ax.plot(s.year, s[m], color=COL[v], lw=2, marker='o', ms=4, ls=ls, label=f'{v.split()[0]} ({"cleaned" if b == "C" else "rebuilt"} base)')
    ax.axhline(0, color='#bdbcb8', lw=0.8); ax.axvspan(2022.5, 2023.5, color='#efefec', zorder=0)
    ax.set_title(t, fontsize=10, loc='left'); ax.set_xticks(YEARS); ax.grid(axis='y', color='#ececea', lw=0.8)
    for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
axes[0].set_ylabel('Δ Spearman ρ'); axes[1].legend(frameon=False, fontsize=7.5)
fig.text(0.99, 0.01, 'shaded: 2023 (suspect, D1)', ha='right', fontsize=7.5, color='#52514e')
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / 'delta_metrics.png', dpi=100); plt.close(fig)
print('saved figures/delta_metrics.png')


saved figures/delta_metrics.png


![delta metrics](figures/delta_metrics.png)

**Takeaway (figure):** DD (blue) is above zero on both metrics in 2019–2021 on the cleaned base (solid). The exceptions are its 2019 M2 dip on the rebuilt base (dotted) and a small 2022 M1 dip on both. A17 (grey) sits below DD on income fit (right panel) in every year.

## Caveats

- **Weak, base-dependent pass.** DD passes rule A on the cleaned base and fails on the rebuilt base by a 2019 M2 margin of 0.006. Every 2019 bootstrap interval includes 0. The primary base was named after a prototype had computed its result (see the Method transparency note), so don't over-read the pass. The consistent part is the *direction* (14 of 16 differences positive) and DD > A17 on M2 in every year.
- **ATLAS is a benchmark, not ground truth**, and M2 penalises resource-rich countries (11-26 G).
- **DD changes many printed ranks.** The median 2019 move is 5 places, and 6 of 28 peer positions change. Any adoption has to happen for all countries in a semester at once, not mid-semester.
- **Binary RCA amplifies everything here.** A uniform per-country rescaling of 0.73–1.36 flips 80 cells, because many cells sit near 1 (11-23). A continuous RCA would respond smoothly.
- **DD doesn't touch D1.** The filtered 17-column world total is still $27,545B in 2023 vs $20,213B in 2022. 2023 remains suspect.
- Nominal values. Saudi Arabia (2022) and Congo (2022–2023) are absent from the raw pulls. The peer dict is the one used by earlier runs (from the Spring 2026 PDFs) and wasn't re-read tonight.

## Helpful to know

- **The aggregate's effect on the denominators reduces to one number per country:** RCA_DD = RCA_D0 × (1 + a_c)/(1 + a_w), with a = aggregate / 17-column total. Any mapping change to the aggregate (11-25's F4, 11-26's reassignment) moves a_c and a_w, which is why those variants flipped cells in unrelated columns.
- **DD = A17's cells + D0's aggregate column, exactly** (2019–2023: 0 aggregate flips, 0.0 difference on the 17 columns). Code-wise, DD is `X.div(X.drop(columns=[AGG]).sum(1), axis=0) / (X.sum(0) / X.drop(columns=[AGG]).to_numpy().sum())`, a one-line change to `compute_mcp`'s `country_sum` and `total_sum`.
- **Under the lab's current D0, the effective RCA threshold differs by country:** about 0.83 for the median country and about 1.16 for Germany (2019). This is a hidden bias against aggregate-heavy exporters.
- The cleaned and rebuilt bases give **identical** results in 2022 and 2023 and differ only in 2019–2021, so base-dependence is a 2019–2021 cleaning issue (Q05).

## New research avenues (also in `proposals/2026-11-27.md`)

1. **DD + F1 together.** F1 (ch. 89 → *Transportation*, 11-24) is the only mapping variant that has passed rule A, and DD now passes weakly on the cleaned base. Score DD+F1 against D0, and against each one alone, on both bases. Is the combination additive, or do they overlap through the same threshold flips? One night.
2. **Why the two bases disagree in 2019.** DD's 2019 ΔM2 is +0.023 on the cleaned base and −0.006 on the rebuilt base. List the countries whose cleaned and rebuilt 2019 rows differ (Q05), and recompute the DD effect with each one swapped between the bases, to find the few that decide the sign. One night.
3. **Continuous RCA with DD denominators.** Repeat tonight's comparison with log(1 + RCA) and capped RCA (Q07 at the grouped level). The DD rescaling is a smooth per-country factor, so a continuous RCA should show whether the gain survives without the 80 threshold flips. One night.
4. **Report-impact memo for DD.** For the 7 Spring 2026 countries, recompute the 2019–2023 grouped-ECI paths and peer sentences under DD and list exactly which printed numbers would change. This is what Will needs before deciding to adopt DD for Fall 2026. Vietnam's 2019 rank (32 → 54) is the headline case. One night.

## Data issues / data needed

None new. Tonight uses only repo data and the already-added external files. The existing D1 (2023 inflation) and 999999-reporter issues (`data-requests/2026-11-25-999999-reporters.md`, in the open 11-25 PR) still apply. DD doesn't fix either.

In [12]:
print(f'Run time: {time.time() - T0:.1f} s')

Run time: 11.2 s
